In [1]:
import numpy as np
import galois as gal

In [48]:
class Poly:
    def __init__(self, coeff, zero=0):
        ''' Class for representing polynomials via lists. The coefficients
        can be any class which has __add__(), sub, __mul__() , _eq__( , 0) and 
        __pow__( , -1) dunder methods. If you want coefficients in regular
        integers or reals or complex, it's much more efficient to use 
        numpy, sympy, etc. instead
        
        self.zero is the zero element of the ring'''
        # trim off excess zeroes
        n = len(coeff)
        if n == 0: # treat empty list as zero polynomial
            coeff.append(zero)
            n = 1 
        is_zero = False
        for c in reversed(coeff):
            if c == zero:
                if n > 1:
                    n -= 1
                    continue
                is_zero = True
                break
            else:
                break          
        self.coeff = coeff[:n]
        self.deg = n-1
        self.is_zero = is_zero
        self.zero = zero

    def __add__(self, other):
        assert self.zero == other.zero, 'need to have the same coefficient field'
        small, big = sorted([self, other], key=lambda x: x.deg)
        sum = [self.coeff[n] + other.coeff[n] for n in range(small.deg + 1)]
        sum += big.coeff[small.deg + 1:]
        return Poly(sum, zero=self.zero)

    def __call__(self, num):
        result = self.zero
        for a in reversed(self.coeff):
            result = result * num
            result = result + a
        return result

    def __eq__(self, other):
        return self.coeff == other.coeff and self.zero == other.zero

    def __floordiv__(self, other):
        assert self.zero == other.zero, 'need to have the same coefficient field'
        numerator = Poly(self.coeff, self.zero) # copy
        quotient = Poly([self.zero], self.zero)
        def monomial(n, const):
            coeff = [self.zero for _ in range(n+1)]
            coeff[n] = const
            return Poly(coeff, self.zero)
        while numerator.deg >= other.deg and not numerator.is_zero:
            coeff = numerator.coeff[-1] * other.coeff[-1] ** -1
            term = monomial(numerator.deg - other.deg, coeff)

            quotient = quotient + term
            numerator = numerator - term * other
        return quotient

    def __getitem__(self, item):
        return self.coeff[item]

    def __iter__(self):
        return self.coeff.__iter__()

    def __mod__(self, other):
        return self - other * (self // other)

    def __mul__(self, other):
        try: # scalar multiplication
            result = [self.coeff[n] * other for n in range(self.deg + 1)]
            return Poly(result, self.zero)
        except:
            deg = self.deg + other.deg
            p, q = self.coeff, other.coeff
            prod = [self.zero for _ in range(deg + 1)]
            for n in range(deg + 1):
                for k in range(n + 1):
                    if k < len(p) and n - k < len(q):
                        prod[n] = prod[n] + p[k] * q[n - k]
            return Poly(prod, zero=self.zero)              

    def __repr__(self):
        return str(self.coeff)
        
    def __sub__(self, other):
        assert self.zero == other.zero, 'need to have the same coefficient field'
        small, big = sorted([self, other], key=lambda x: x.deg)
        pad = [small.coeff[n] for n in range(small.deg + 1)]
        pad += [self.zero for _ in range(big.deg - small.deg)]
        if self.deg <= other.deg: 
            diff = [pad[n] - other.coeff[n] for n in range(len(pad))]
        else:
            diff = [self.coeff[n] - pad[n] for n in range(len(pad))]
        return Poly(diff, zero=self.zero)


In [73]:
class Z:
    def __init__(self, mod : int, num : int = 0, ):
        ''' A class for representing finite cyclic groups.'''
        assert type(mod) == int and mod > 0, 'modulus is positive integer'

        self.repr = num % mod
        self.mod = mod
        self.inv = ... # sentinel since `None` will mean not inverible
    
    def __add__(self, other) -> object:
        assert self.mod == other.mod, 'adding in cyclic group '\
                                      'needs to have same modulus'
        result = self.repr + other.repr % self.mod
        return Z(self.mod, result)
    
    def __eq__(self, other) -> bool:
        try: # if other is also a Z(n) object
            if self.repr == other.repr and self.mod == other.mod:
                return True
            else:
                return False
        except: # allow comparison to integer
            if type(other) == int: 
                return (self.repr - other % self.mod) == 0
            else: # all others return False
                return False
    
    def __mul__(self, other) -> object:
        try:
            if self.mod == other.mod:
                return Z(self.mod, self.repr * other.repr % self.mod)
            else:
                print('need to have same modulus to multiply')
                return None
        except:
            assert type(other) == int, 'multiplication is supported with another'\
            'element of cyclic group or with an integer'
            return Z(self.mod, self.repr * other % self.mod)

    def __pow__(self, power : int) -> object:
        return Z(self.mod, pow(self.repr, power, self.mod))

    def __repr__(self):
        return str(self.repr)
    
    def __sub__(self, other):
        return Z(self.mod, self.repr - other.repr % self.mod)

    def inverse(self) -> int | None:
        ''' Returns the inverse if it exists or None if it doesn't'''
        if self.inv == ... : # not already computed
            try:
                self.inv = pow(self.repr, -1, self.mod)
            except:
                self.inv = None
        return self.inv
    
class PolyMod(Poly):
    ''' Polynomials with coefficients in finite cyclid group'''
    def __init__(self, data : list[int] | int, mod : int):
        if type(data) == list:
            mod_coeff = [Z(mod, n) for n in data]
        elif type(data) == int:
            assert data >= 0, 'integer input should be non-negative'
            def base_decomp(base : int, num : int) -> list[int]:
                ''' write an integer in a given base'''
                decomp = []
                while num > 0:
                    decomp.append(num % base)
                    num //= base
                return decomp
            mod_coeff = [Z(mod, n) for n in base_decomp(mod, data)]
        else:
            raise ValueError('Input is either list or integer')
        zero = Z(mod, 0)
        super().__init__(mod_coeff, zero)
        reduced_coeff = [c.repr for c in mod_coeff]

        self.id = Poly(reduced_coeff)(mod)
        self.mod = mod
    
    def __add__(self, other):
        assert self.mod == other.mod, 'need same base field'
        sum = super().__add__(other)
        coeff = [c.repr for c in sum.coeff]
        return PolyMod(coeff, self.mod)
    
    def __floordiv__(self, other):
        assert self.mod == other.mod, 'need same base field'
        div = super().__floordiv__(other)
        coeff = [c.repr for c in div.coeff]
        return PolyMod(coeff, self.mod)
    
    def __mod__(self, other):
        assert self.mod == other.mod, 'need same base field'
        remainder = super().__mod__(other)
        coeff = [c.repr for c in remainder.coeff]
        return PolyMod(coeff, self.mod)
    
    def __mul__(self, other):
        assert self.mod == other.mod, 'need same base field'
        prod = super().__mul__(other)
        coeff = [c.repr for c in prod.coeff]
        return PolyMod(coeff, self.mod)
    
    def __sub__(self, other):
        assert self.mod == other.mod, 'need same base field'
        sub = super().__sub__(other)
        coeff = [c.repr for c in sub.coeff]
        return PolyMod(coeff, self.mod)       



In [23]:
zero, one = Z(2, 0), Z(2, 1)
p = Poly([one, one, one], zero)
q = Poly([one, one], zero)
q * q, p //q, p(one)

([1, 0, 1], [0, 1], 1)